In [2]:
# ============================================================
# Cell 1: Import Required Libraries + Reproducibility Seed
# ============================================================

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

SEED = 2024  # <-- change this to 123 and 2024 for the 2nd and 3rd run

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cell 1 : Libraries Imported")
print(f"Seed   : {SEED}")
print(f"Device : {DEVICE}")

Cell 1 : Libraries Imported
Seed   : 2024
Device : cuda


In [3]:

# ============================================================
# Cell 2: Path Configuration & Hyperparameters
# ============================================================
PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")

DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"
TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

assert TRAIN_DIR.exists(), f"Train dir not found: {TRAIN_DIR}"
assert VALID_DIR.exists(), f"Validation dir not found: {VALID_DIR}"
assert TEST_DIR.exists(), f"Test dir not found: {TEST_DIR}"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 260
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0
CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

MODEL_ID = f"pneumoxnet_seed{SEED}"        # <-- seed-wise unique filename, purono checkpoint overwrite hobe na
BEST_MODEL_PATH = MODELS_DIR / f"{MODEL_ID}_best_acc.pth"
BEST_LOSS_MODEL_PATH = MODELS_DIR / f"{MODEL_ID}_best_loss.pth"
HISTORY_PATH = RESULTS_DIR / f"{MODEL_ID}_history.csv"

print("Cell 2 : Config Set")
print(f"Model ID : {MODEL_ID}")

Cell 2 : Config Set
Model ID : pneumoxnet_seed2024


In [4]:
# ============================================================
# Cell 3: Data Augmentation & Preprocessing (Stronger Regularization)
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),                              # 15 -> 20
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.85, 1.15)),  # range barano
    transforms.ColorJitter(brightness=0.2, contrast=0.2),                # 0.15 -> 0.2
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.35, scale=(0.02, 0.15)),                # 0.25 -> 0.35, scale barano
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Cell 3 : Transforms Ready (Stronger Augmentation)")

Cell 3 : Transforms Ready (Stronger Augmentation)


In [5]:
# ============================================================
# Cell 4: Dataset Loading and DataLoaders
# ============================================================

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = datasets.ImageFolder(VALID_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_dataset.targets),
    y=train_dataset.targets
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Cell 4 : Dataset Loaded")
print(f"Classes         : {train_dataset.classes}")
print(f"Class to idx    : {train_dataset.class_to_idx}")
print(f"Train / Valid / Test : {len(train_dataset)} / {len(valid_dataset)} / {len(test_dataset)}")

Cell 4 : Dataset Loaded
Classes         : ['BACTERIA', 'NORMAL', 'VIRUS']
Class to idx    : {'BACTERIA': 0, 'NORMAL': 1, 'VIRUS': 2}
Train / Valid / Test : 4099 / 878 / 879


In [6]:
# ============================================================
# Cell 5: Architectural Building Blocks
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))


class MultiScaleFeatureFusion(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        mid = in_channels // reduction
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        self.branch_1x1 = nn.Sequential(
            nn.Conv2d(mid, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        self.branch_3x3 = nn.Sequential(
            nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        self.branch_5x5 = nn.Sequential(
            nn.Conv2d(mid, mid, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(mid * 3, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.reduce(x)
        f = torch.cat([self.branch_1x1(x), self.branch_3x3(x), self.branch_5x5(x)], dim=1)
        return self.fusion(f)


class AdaptiveFeatureFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight_generator = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 1, bias=False), nn.Sigmoid()
        )

    def forward(self, feature_a, feature_b):
        weights = self.weight_generator(torch.cat([feature_a, feature_b], dim=1))
        return weights * feature_a + (1.0 - weights) * feature_b


class ResidualEnhancement(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return x + self.refine(x)


print("Cell 5 : Building Blocks Defined (CBAM, MultiScaleFeatureFusion, AdaptiveFeatureFusion, ResidualEnhancement)")

Cell 5 : Building Blocks Defined (CBAM, MultiScaleFeatureFusion, AdaptiveFeatureFusion, ResidualEnhancement)


In [7]:
# ============================================================
# Cell 6: PneumoXNet Full Architecture
# ============================================================

class PneumoXNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        weights = EfficientNet_B2_Weights.DEFAULT
        backbone = efficientnet_b2(weights=weights)
        self.backbone = backbone.features
        self.feature_channels = 1408

        self.cbam = CBAM(self.feature_channels)
        self.multiscale = MultiScaleFeatureFusion(self.feature_channels)
        self.aff = AdaptiveFeatureFusion(self.feature_channels)
        self.residual = ResidualEnhancement(self.feature_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.55),                      # 0.5 -> 0.55
            nn.Linear(self.feature_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.45),                      # 0.4 -> 0.45
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        cbam_out = self.cbam(features)
        ms_out = self.multiscale(features)
        fused = self.aff(cbam_out, ms_out)
        enhanced = self.residual(fused)
        pooled = self.pool(enhanced)
        return self.classifier(pooled)


model = PneumoXNet(num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 70)
print("Cell 6 : PneumoXNet Architecture Initialized (Stronger Dropout)")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print("=" * 70)

Cell 6 : PneumoXNet Architecture Initialized (Stronger Dropout)
Total Parameters     : 36,809,319
Trainable Parameters : 36,809,319


In [8]:
# ============================================================
# Cell 7: Training Configuration (Loss / Optimizer / Scheduler)
# ============================================================

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

new_module_params = (
    list(model.cbam.parameters())
    + list(model.multiscale.parameters())
    + list(model.aff.parameters())
    + list(model.residual.parameters())
    + list(model.classifier.parameters())
)

optimizer = optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": LEARNING_RATE * 0.7, "weight_decay": 3e-4},   # 2e-4 -> 3e-4
        {"params": new_module_params, "lr": LEARNING_RATE, "weight_decay": 7e-4}                     # 5e-4 -> 7e-4
    ]
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6            # patience 4 -> 3 (kom epoch-e adjust)
)

print("Cell 7 : Loss/Optimizer/Scheduler Ready (Stronger Weight Decay)")

Cell 7 : Loss/Optimizer/Scheduler Ready (Stronger Weight Decay)


In [9]:
# ============================================================
# Cell 8: Train / Validate / Test Functions
# ============================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total


def evaluate_test_set(model, dataloader, device, class_names):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, macro_f1, report


print("Cell 8 : Train/Validate/Test Functions Ready")

Cell 8 : Train/Validate/Test Functions Ready


In [10]:
history = {"epoch": [], "train_loss": [], "train_accuracy": [], "valid_loss": [], "valid_accuracy": [], "learning_rate": []}

best_val_accuracy = 0.0
best_val_loss = float("inf")
best_epoch = 0
best_loss_epoch = 0

early_stopping_patience = 7          # 10 -> 7 (25 epoch-er jonno proportionally kom)
early_stopping_counter = 0

start_time = time.time()

print("Cell 9 : Training Initialized")

Cell 9 : Training Initialized


In [11]:
# ============================================================
# Cell 10: PneumoXNet Training Loop
# ============================================================

print("=" * 70)
print("Training PneumoXNet")
print("=" * 70)

for epoch in range(EPOCHS):

    epoch_start_time = time.time()
    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")
    print("-" * 70)

    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    valid_loss, valid_accuracy = validate_one_epoch(model, valid_loader, criterion, DEVICE)

    scheduler.step(valid_accuracy)
    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["valid_loss"].append(valid_loss)
    history["valid_accuracy"].append(valid_accuracy)
    history["learning_rate"].append(current_lr)

    if valid_accuracy > best_val_accuracy:
        best_val_accuracy = valid_accuracy
        best_epoch = epoch + 1
        early_stopping_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Best model (by accuracy) updated.")
    else:
        early_stopping_counter += 1

    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        best_loss_epoch = epoch + 1
        torch.save(model.state_dict(), BEST_LOSS_MODEL_PATH)
        print("Best model (by loss) updated.")

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    epoch_time = time.time() - epoch_start_time
    print(f"Train Loss : {train_loss:.4f}  Train Acc : {train_accuracy:.4f}")
    print(f"Valid Loss : {valid_loss:.4f}  Valid Acc : {valid_accuracy:.4f}")
    print(f"LR : {current_lr:.6f}  Time : {epoch_time:.2f}s")
    print(f"Best Acc : {best_val_accuracy:.4f} (epoch {best_epoch})  Best Loss : {best_val_loss:.4f} (epoch {best_loss_epoch})")

    if early_stopping_counter >= early_stopping_patience:
        print("\nEarly stopping triggered.")
        break

training_time = time.time() - start_time
print("\n" + "=" * 70)
print("Training Completed")
print(f"Best Epoch (Acc)  : {best_epoch}  |  Best Val Accuracy : {best_val_accuracy:.4f}")
print(f"Best Epoch (Loss) : {best_loss_epoch}  |  Best Val Loss : {best_val_loss:.4f}")
print(f"Total Time : {training_time / 60:.2f} min")
print("=" * 70)

Training PneumoXNet

Epoch [1/20]
----------------------------------------------------------------------
Best model (by accuracy) updated.
Best model (by loss) updated.
Train Loss : 0.7755  Train Acc : 0.7070
Valid Loss : 0.6826  Valid Acc : 0.7973
LR : 0.000070  Time : 142.08s
Best Acc : 0.7973 (epoch 1)  Best Loss : 0.6826 (epoch 1)

Epoch [2/20]
----------------------------------------------------------------------
Best model (by accuracy) updated.
Best model (by loss) updated.
Train Loss : 0.6992  Train Acc : 0.7702
Valid Loss : 0.6699  Valid Acc : 0.8087
LR : 0.000070  Time : 144.14s
Best Acc : 0.8087 (epoch 2)  Best Loss : 0.6699 (epoch 2)

Epoch [3/20]
----------------------------------------------------------------------
Train Loss : 0.6637  Train Acc : 0.7865
Valid Loss : 0.7001  Valid Acc : 0.7597
LR : 0.000070  Time : 139.45s
Best Acc : 0.8087 (epoch 2)  Best Loss : 0.6699 (epoch 2)

Epoch [4/20]
----------------------------------------------------------------------
Train Lo

In [12]:
# ============================================================
# Cell 11: Load Best Model and Evaluate on Test Set
# ============================================================

model.load_state_dict(torch.load(BEST_MODEL_PATH))

test_acc, test_f1, report = evaluate_test_set(model, test_loader, DEVICE, CLASS_NAMES)

print("=" * 70)
print("PneumoXNet — Final Test Set Results (Best-by-Accuracy checkpoint)")
print("=" * 70)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Macro-F1 : {test_f1:.4f}")
print("-" * 70)
print(report)

model.load_state_dict(torch.load(BEST_LOSS_MODEL_PATH))
test_acc_loss, test_f1_loss, report_loss = evaluate_test_set(model, test_loader, DEVICE, CLASS_NAMES)

print("\n" + "=" * 70)
print("PneumoXNet — Final Test Set Results (Best-by-Loss checkpoint)")
print("=" * 70)
print(f"Test Accuracy : {test_acc_loss:.4f}")
print(f"Test Macro-F1 : {test_f1_loss:.4f}")
print("-" * 70)
print(report_loss)

PneumoXNet — Final Test Set Results (Best-by-Accuracy checkpoint)
Test Accuracy : 0.8237
Test Macro-F1 : 0.8197
----------------------------------------------------------------------
              precision    recall  f1-score   support

    BACTERIA     0.8707    0.7914    0.8291       417
      NORMAL     0.8996    0.9790    0.9376       238
       VIRUS     0.6680    0.7188    0.6925       224

    accuracy                         0.8237       879
   macro avg     0.8128    0.8297    0.8197       879
weighted avg     0.8269    0.8237    0.8237       879


PneumoXNet — Final Test Set Results (Best-by-Loss checkpoint)
Test Accuracy : 0.8237
Test Macro-F1 : 0.8197
----------------------------------------------------------------------
              precision    recall  f1-score   support

    BACTERIA     0.8707    0.7914    0.8291       417
      NORMAL     0.8996    0.9790    0.9376       238
       VIRUS     0.6680    0.7188    0.6925       224

    accuracy                         0

In [13]:
# ============================================================
# Cell 12: Log This Seed's Result for Later Aggregation
# ============================================================

RESULT_LOG_PATH = RESULTS_DIR / "pneumoxnet_multiseed_results.csv"

result_row = {
    "seed": SEED,
    "best_epoch": best_epoch,
    "test_accuracy": test_acc,
    "test_macro_f1": test_f1
}

if RESULT_LOG_PATH.exists():
    log_df = pd.read_csv(RESULT_LOG_PATH)
    log_df = pd.concat([log_df, pd.DataFrame([result_row])], ignore_index=True)
else:
    log_df = pd.DataFrame([result_row])

log_df.to_csv(RESULT_LOG_PATH, index=False)

print("=" * 70)
print(f"Result logged for seed {SEED}")
print(log_df.to_string(index=False))
print("=" * 70)

if len(log_df) >= 3:
    print(f"\nMean Accuracy : {log_df['test_accuracy'].mean():.4f} ± {log_df['test_accuracy'].std():.4f}")
    print(f"Mean Macro-F1 : {log_df['test_macro_f1'].mean():.4f} ± {log_df['test_macro_f1'].std():.4f}")

Result logged for seed 2024
 seed  best_epoch  test_accuracy  test_macro_f1
   42          10       0.832765       0.830283
  123          15       0.847554       0.835762
 2024           5       0.823663       0.819748

Mean Accuracy : 0.8347 ± 0.0121
Mean Macro-F1 : 0.8286 ± 0.0081


In [14]:
import pandas as pd
from pathlib import Path

RESULT_LOG_PATH = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/results/pneumoxnet_multiseed_results.csv")

log_df = pd.read_csv(RESULT_LOG_PATH)
log_df = log_df.drop_duplicates(subset="seed", keep="first")
log_df.to_csv(RESULT_LOG_PATH, index=False)

print(log_df)

   seed  best_epoch  test_accuracy  test_macro_f1
0    42          10       0.832765       0.830283
1   123          15       0.847554       0.835762
2  2024           5       0.823663       0.819748
